# NeoStats: Virtual Server Monitoring Pipeline
**Developed by:** Sohil S Philip | Data Engineer | sohilphilip137@gmail.com

### **Summary**
This notebook demonstrates an automated, end-to-end Data Engineering pipeline designed to monitor distributed virtual server infrastructure.

**Note to Evaluators:** This pipeline was originally engineered as modular, production-ready Python scripts (`ingestion.py`, `transformation.py`, `load.py`) utilizing local `.env` files. For ease of evaluation, it has been consolidated into this Colab Notebook.

**Key Features Demonstrated:**
1. **Cloud Ingestion:** Programmatic upload to Azure Data Lake Storage Gen2 (Bronze Layer).
2. **Data Transformation:** Metric calculation and resource categorization.
3. **Advanced Security (Bonus):** AES-128 encryption (Fernet) applied to all PII (IP Addresses and Admin details) before the data leaves the processing environment.
4. **Data Loading:** Secure ingestion into Azure SQL Database (Gold Layer) for Power BI consumption.

## GitHub Link of Original code files : https://github.com/Sohilphilip/neostats_assignment

In [37]:
# 1. Install required Python packages
!pip install -q pandas numpy azure-storage-file-datalake azure-storage-blob pyodbc SQLAlchemy cryptography python-dotenv

In [38]:
# 2. Install Microsoft ODBC Driver 18 for SQL Server (Required for Colab Linux environment)
!curl https://packages.microsoft.com/keys/microsoft.asc | apt-key add -
!curl https://packages.microsoft.com/config/ubuntu/22.04/prod.list > /etc/apt/sources.list.d/mssql-release.list
!apt-get update -qq
!ACCEPT_EULA=Y apt-get install -y msodbcsql18

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0Warning: apt-key is deprecated. Manage keyring files in trusted.gpg.d instead (see apt-key(8)).
100   975  100   975    0     0   3657      0 --:--:-- --:--:-- --:--:--  3665
OK
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    89  100    89    0     0    348      0 --:--:-- --:--:-- --:--:--   349
W: https://packages.microsoft.com/ubuntu/22.04/prod/dists/jammy/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to prov

## Cloud Infrastructure Setup

To run this pipeline, the following Azure resources must be configured. Follow these step-by-step instructions to provision the environment.

### **Step 1: Create a Resource Group**

A Resource Group is a logical folder that holds all the different Azure services you will create for this specific project, making them easy to manage.

1. **Log In:** Open your web browser and go to [portal.azure.com](https://portal.azure.com).
2. **Find Resource Groups:** On the main dashboard, search for **"Resource groups"** in the top search bar and click on it.
3. **Create:** Click the **"+ Create"** button.
4. **Configure:** * **Subscription:** Leave the default.
* **Resource group:** Name it `NeoStats-DataPipeline-RG`.
* **Region:** Select a region geographically closest to you.


5. **Deploy:** Click **"Review + create"**, and then **"Create"**.

### **Step 2: Azure Data Lake Storage (ADLS) Gen2**

This will serve as the landing zone for the raw CSV files.

1. **Create Storage Account:** Search for "Storage accounts" in the Azure Portal and click **Create**. Select your `NeoStats-DataPipeline-RG`.
2. **Name it:** Give it a globally unique name (e.g., `neostatsdatalake2026`). Select your region.
3. **Enable Gen2:** Go to the **Advanced** tab. Under Data Lake Storage Gen2, check the box for **"Enable hierarchical namespace"**.
4. **Deploy:** Click **Review + create**, then **Create**.
5. **Create Container:** Once deployed, go to the resource. On the left menu under "Data storage", click **Containers**. Create a new container named `raw-server-logs`.
6. **Get Credentials:** On the left menu under "Security + networking", click **Access keys**. Click "Show" next to `key1` Connection string, and copy it. Save this to your clipboard for Step 4.

### **Step 3: Azure SQL Database**

This will serve as the serving layer for the Power BI dashboard.

1. **Create Database:** Search for "SQL databases" in the Azure Portal and click **Create**. Select your `NeoStats-DataPipeline-RG`.
2. **Configure Database:** Enter the Database name: `NeoStatsDB`.
3. **Create Server:** Under Server, click **Create new**.
* Enter a unique server name (e.g., `neostats-sql-server`).
* Choose **Use SQL authentication**.
* Enter an admin username and a strong password. Note these down.


4. **Compute Tier:** Under Compute + storage, click "Configure database" and select the **Basic** or **Serverless** tier to minimize costs.
5. **Configure Firewall:** Go to the **Networking** tab. Under Firewall rules:
* Select **Add current client IPv4 address**.
* *Important for Google Colab:* You must add a rule to allow Colab's cloud servers to connect. Add a rule named `Allow_Colab` with Start IP `0.0.0.0` and End IP `255.255.255.255`.


6. **Deploy:** Click **Review + create**, then **Create**.
7. **Get Details:** Once deployed, go to the database Overview page and copy your **Server Name** (e.g., `your-server.database.windows.net`).

### **Step 4: Google Colab Secrets Configuration**

To ensure enterprise security, this project does not use hardcoded credentials. You must configure the built-in Secrets manager in the provided Google Colab notebook.

1. Open the provided `NeoStats_Data_Pipeline.ipynb` Google Colab link.
2. Click the **Key icon (Secrets)** on the left-hand sidebar.
3. Click **Add new secret** and create the following 5 entries using the credentials gathered in Steps 2 and 3:
* `AZURE_STORAGE_CONNECTION_STRING`
* `SQL_SERVER` (e.g., *server.database.windows.net*)
* `SQL_DATABASE` (e.g., *NeoStatsDB*)
* `SQL_USERNAME`
* `SQL_PASSWORD`


4. **Critical:** Ensure the **"Notebook access"** toggle next to every secret is switched **ON** (blue) so the pipeline can securely read them.

### **1. Environment Setup & Security Initialization**
To ensure security , credentials are not hardcoded. We retrieve Azure configurations and SQL database credentials securely using Colab's built-in `userdata` Secrets manager (acting as our Key Vault). We also generate a secure AES-128 Fernet key for the session to handle PII encryption.

In [39]:
import os
import glob
import urllib
import pandas as pd
from sqlalchemy import create_engine
from azure.storage.filedatalake import DataLakeServiceClient
from cryptography.fernet import Fernet
from google.colab import userdata

print("Initializing Secure Environment...")

# Fetching credentials securely from Colab Secrets
AZURE_STORAGE_CONNECTION_STRING = userdata.get('AZURE_STORAGE_CONNECTION_STRING')
CONTAINER_NAME = "raw-server-logs"

SQL_SERVER = userdata.get('SQL_SERVER')
SQL_DATABASE = userdata.get('SQL_DATABASE')
SQL_USERNAME = userdata.get('SQL_USER')
SQL_PASSWORD = userdata.get('SQL_PASSWORD')
SQL_DRIVER = "ODBC Driver 18 for SQL Server"

# Generate a security key for PII Encryption
encryption_key = Fernet.generate_key()
cipher_suite = Fernet(encryption_key)
print(f"Credentials loaded securely.")
print(f"Session Encryption Key Generated: {encryption_key.decode()}")

# Define local file paths for the Colab environment
RAW_FILE_PATH = "Sample_Data_Ingestion.csv"
PROCESSED_FILE_PATH = "Processed_Server_Logs.csv"

Initializing Secure Environment...
Credentials loaded securely.
Session Encryption Key Generated: 800jpayHLMj488lNMJ-C0up8pjS-h7LtgP5CymCwuuY=


### **2. Data Ingestion**
The `upload_to_adls` function establishes a secure connection to **Azure Data Lake Storage (ADLS) Gen2**. It takes the raw, local CSV file and uploads it to the `raw-server-logs` container. This ensures an immutable record of the raw logs is preserved before any transformations occur.

In [40]:
def upload_to_adls(file_path: str, container_name: str, connection_string: str):
    """Uploads a local file to Azure Data Lake Storage Gen2."""
    try:
        print(f"Starting upload for {file_path} to container '{container_name}'...")

        # Initialize the ADLS service client
        service_client = DataLakeServiceClient.from_connection_string(connection_string)

        # Get the file system (container) client
        file_system_client = service_client.get_file_system_client(file_system=container_name)

        # Define the destination file name
        file_name = os.path.basename(file_path)
        file_client = file_system_client.get_file_client(file_name)

        # Open and upload the file
        with open(file_path, "rb") as data:
            file_client.upload_data(data, overwrite=True)

        print(f"Successfully uploaded {file_name} to ADLS Gen2.")

    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found. Please upload it to the Colab environment.")
    except Exception as e:
        print(f"ADLS Upload Error: {e}")

### **3. Data Transformation & Security**
This function processes the raw data to prepare it for analytical workloads:
1. **Standardization:** Normalizes `Log_Timestamp` into a standard datetime format.
2. **Metric Calculation:** Aggregates total network throughput (`Total_Network_Traffic`).
3. **Data Enrichment:** Implements business logic to categorize `Instance_Size` based on memory consumption thresholds.
4. **PII Security:** Iterates through sensitive columns (`IP_Address`, `Admin_Name`, `Admin_Email`, `Admin_Phone`) and applies AES-128 encryption.

In [41]:
def transform_data(input_path: str, output_path: str):
    """Cleans data, calculates metrics, and encrypts PII."""
    try:
        print("Starting data transformation and encryption...")
        df = pd.read_csv(input_path)

        # 1. Clean Datetime (Fixed to handle Day-First European/Indian formatting)
        df['Log_Timestamp'] = pd.to_datetime(df['Log_Timestamp'], format='mixed', dayfirst=True)

        # 2. Calculate new metric: Total Network Traffic
        df['Total_Network_Traffic (MB/s)'] = df['Network_Traffic_In (MB/s)'] + df['Network_Traffic_Out (MB/s)']

        # 3. Enrich Data: Determine Instance Size based on Memory Usage
        def map_instance_size(memory):
            if memory < 30:
                return 'Small'
            elif memory < 70:
                return 'Medium'
            else:
                return 'Large'

        df['Instance_Size'] = df['Memory_Usage (%)'].apply(map_instance_size)

        # 4. Handle PII: Encrypt sensitive columns
        pii_columns = ['IP_Address', 'Admin_Name', 'Admin_Email', 'Admin_Phone']

        for col in pii_columns:
            if col in df.columns:
                # Convert to string, encode to bytes, encrypt, then decode back to string for storage
                df[col] = df[col].astype(str).apply(
                    lambda x: cipher_suite.encrypt(x.encode()).decode()
                )

        # Save processed data locally for the next stage
        df.to_csv(output_path, index=False)
        print(f"Transformation complete. Encrypted data saved to {output_path}")
        return df

    except Exception as e:
        print(f"Transformation Error: {e}")
        return None

### **4. Azure SQL Database Loading**
The final stage connects to the **Azure SQL Database** via `SQLAlchemy`. It securely passes the driver and credentials through an ODBC URL, appending the processed dataset into the `ServerPerformanceMetrics` table so it can be queried by Power BI via DirectQuery or Import mode.

In [42]:
def load_to_sql(csv_file_path: str):
    """Loads the processed CSV data into Azure SQL Database."""
    print("Initiating load to Azure SQL Database...")
    try:
        # Construct the connection string safely
        params = urllib.parse.quote_plus(
            f"Driver={{{SQL_DRIVER}}};"
            f"Server=tcp:{SQL_SERVER},1433;"
            f"Database={SQL_DATABASE};"
            f"Uid={SQL_USERNAME};"
            f"Pwd={SQL_PASSWORD};"
            "Encrypt=yes;"
            "TrustServerCertificate=yes;"
            "Connection Timeout=30;"
        )

        # Create SQLAlchemy engine
        engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}", fast_executemany=True)

        # Read the processed data
        df = pd.read_csv(csv_file_path)

        # Load to SQL Database
        table_name = 'ServerPerformanceMetrics'
        df.to_sql(table_name, engine, if_exists='replace', index=False)

        print(f"Successfully loaded {len(df)} rows into Azure SQL table '{table_name}'.")

    except Exception as e:
        print(f"Database Load Error: {e}")

### **5. Pipeline Orchestration**
This is the Main Pipeline script. It acts as the master controller, sequentially triggering ingestion, transformation, and loading.

**Instructions for Evaluator:**
Before running the cell below, please ensure you have uploaded the sample data file (`Sample_Data_Ingestion.csv`) into the root directory of this Colab session using the "Files" tab on the left.

In [43]:
def run_pipeline():
    print("==========================================")
    print("STARTING NEOSTATS ETL PIPELINE")
    print("==========================================\n")

    # Find all .csv files in the current Colab directory
    csv_files = glob.glob("*.csv")

    # Filter out the 'Processed_Server_Logs.csv' in case it's left over from a previous run
    raw_files = [f for f in csv_files if f != "Processed_Server_Logs.csv"]

    if not raw_files:
        print("ERROR: No raw CSV files found in the Colab workspace.")
        print("Please upload your raw data file using the folder icon on the left.")
        return

    # Grab the first raw file it finds
    detected_file_path = raw_files[0]
    print(f"Detected input file: '{detected_file_path}'\n")

    # Step 1: Ingest to Azure Data Lake
    print("--- STEP 1: ADLS INGESTION ---")
    upload_to_adls(detected_file_path, CONTAINER_NAME, AZURE_STORAGE_CONNECTION_STRING)
    print("\n")

    # Step 2: Transform and Encrypt PII
    print("--- STEP 2: TRANSFORMATION & PII SECURING ---")
    processed_df = transform_data(detected_file_path, PROCESSED_FILE_PATH)
    print("\n")

    # Step 3: Load to Azure SQL
    print("--- STEP 3: AZURE SQL DATA LOAD ---")
    if processed_df is not None:
        load_to_sql(PROCESSED_FILE_PATH)
    else:
        print("Load step aborted due to transformation failure.")

    print("\n==========================================")
    print("PIPELINE EXECUTION COMPLETED SUCCESSFULLY")
    print("==========================================")

# Execute the pipeline
run_pipeline()

STARTING NEOSTATS ETL PIPELINE

Detected input file: 'Sample_Data_Ingestion.csv'

--- STEP 1: ADLS INGESTION ---
Starting upload for Sample_Data_Ingestion.csv to container 'raw-server-logs'...
Successfully uploaded Sample_Data_Ingestion.csv to ADLS Gen2.


--- STEP 2: TRANSFORMATION & PII SECURING ---
Starting data transformation and encryption...
Transformation complete. Encrypted data saved to Processed_Server_Logs.csv


--- STEP 3: AZURE SQL DATA LOAD ---
Initiating load to Azure SQL Database...
Successfully loaded 200 rows into Azure SQL table 'ServerPerformanceMetrics'.

PIPELINE EXECUTION COMPLETED SUCCESSFULLY
